# Clinical-Longformer vs. DistilBERT vs. the production model

This notebook fine-tunes **Clinical-Longformer** (the model Christopher used) on the same synthetic MIDAS-style dataset and the same 3-bucket task (`none` / `some` / `serious`) as the local runs. Data loading goes through `data_pipeline.py`, so it gets the same leakage-safe input (intake fields only, no `manager_comments` / `unit_actions_taken`) and the same date-based split (January–August train, September validation, October test). The results are directly comparable to the local numbers.

**Before running:** In Colab, go to `Runtime > Change runtime type` and set Hardware accelerator to **GPU** (T4 is fine).

In [ ]:
!pip install -q transformers datasets evaluate accelerate pyarrow

## Upload the data and the pipeline
Run this cell, then choose **both** `LOCAL_ONLY_student_facing_candidate.parquet` (from Downloads > "Capstone Engineering Files") and `data_pipeline.py` (from the repo).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select the parquet file AND data_pipeline.py
DATA_PATH = next(name for name in uploaded if name.endswith('.parquet'))
assert 'data_pipeline.py' in uploaded, 'Upload data_pipeline.py too'
print('Using file:', DATA_PATH)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from data_pipeline import train_val_test, LABELS, LABEL2ID

MODEL_NAME = "yikuan8/Clinical-Longformer"
MAX_LENGTH = 512  # our narratives are short (~400 chars avg); this is generous headroom
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

# Leakage-safe input text and the date-based split, shared with every local run.
# build_input_text() raises if a post-review column ever reaches the model.
train_df, val_df, test_df = train_val_test(DATA_PATH)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

for split_df in (train_df, val_df, test_df):
    split_df["label"] = split_df["bucket"].map(LABEL2ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device != "cuda":
    print("WARNING: no GPU detected -- go to Runtime > Change runtime type > GPU")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True)).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True)).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True)).map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

keep_cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=keep_cols)
val_ds.set_format(type="torch", columns=keep_cols)
test_ds.set_format(type="torch", columns=keep_cols)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

counts = train_df["label"].value_counts().reindex(range(len(LABELS)), fill_value=0).to_numpy(dtype=np.float32)
inv_freq = 1.0 / np.where(counts == 0, 1e-6, counts)
class_weights = inv_freq / inv_freq.sum() * len(LABELS)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print("Class weights:", dict(zip(LABELS, class_weights.tolist())))

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    report = classification_report(labels, preds, target_names=LABELS, output_dict=True, zero_division=0)
    return {
        "accuracy": report["accuracy"],
        "serious_recall": report["serious"]["recall"],
        "serious_precision": report["serious"]["precision"],
        "macro_f1": report["macro avg"]["f1-score"],
    }

training_args = TrainingArguments(
    output_dir="./longformer_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="serious_recall",
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
)

trainer.train()

In [ ]:
test_results = trainer.predict(test_ds)
logits = test_results.predictions
labels = test_results.label_ids
preds = np.argmax(logits, axis=1)

print("=== Clinical-Longformer on held-out test set ===")
print(classification_report(labels, preds, target_names=LABELS, zero_division=0))
cm = confusion_matrix(labels, preds, labels=list(range(len(LABELS))))
print("Confusion matrix (rows=true, cols=pred), label order:", LABELS)
print(cm)

## Compare against what we already have (from local runs)

October test set (8,312 reports: 7,909 none, 383 some, 20 serious), same split and inputs as above. Local numbers are from `severity_model/evaluation.json` and the DistilBERT rerun in `PIPELINE.md`.

| Precision / recall | Production severity model | DistilBERT (128 tokens) | Clinical-Longformer |
|---|---|---|---|
| none | 0.99 / 0.99 | 0.99 / 0.97 | *(fill in)* |
| some | 0.76 / 0.83 | 0.55 / 0.85 | *(fill in)* |
| serious | 0.62 / 0.75 | 0.78 / 0.70 | *(fill in)* |
| Macro F1 | 0.82 | 0.79 | *(fill in)* |

Paste the numbers from the cell above into the last column.